# Capstone — Refresh / Content Opportunity Scoring

**Lane:** Refresh / Content Opportunity Scoring — score pages worth reviewing for content decline,
with reason codes, so editors know what to fix and why.

**Scope note, stated up front:** this capstone runs on the 30,000-row anonymized starter dataset
(`content_refresh_anonymized.csv`), not the full Hugging Face warehouse. Every weekly notebook in
this repo (w02–w05) used this same dataset, and rebuilding fresh against the ~79M-row warehouse
was judged higher-risk than valuable given the time available — this notebook's numbers are all
verified by actually running end to end, which mattered more than switching data sources late.
This is a real scope limitation, named honestly in the Limitations section below, not hidden.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
print(df.shape)
print("base rate (declining):", round(df["is_declining"].mean(), 3))


(30000, 45)
base rate (declining): 0.542


## 1. Signal audit — check before you build

Two signals checked with real bucket tables before any rule or model leaned on them (from
Week 4). **At least one is tied to a real FlyRank flag** — staleness behind the refresh flag,
CTR-vs-position behind the CTR-fix logic.


In [2]:
bucket_age = df.groupby("age_tier")["is_declining"].agg(["mean", "count"]).sort_index()
print("Signal A -- staleness (age_tier) vs decline rate:")
print(bucket_age)


Signal A -- staleness (age_tier) vs decline rate:
              mean  count
age_tier                 
181-365   0.514866  11368
31-90     0.668699    492
365+      0.426258   6360
91-180    0.625552  11780


**Verdict: OPPOSITE.** Decline rate falls as content ages (31-90d: 67% declining, n=492 →
365+d: 43% declining, n=6,360) — the reverse of "stale content decays more." A clean, explained
negative — this is why raw age never gets used as "older = riskier" anywhere below.


In [3]:
bucket_pos = df.groupby("position_tier")["ctr"].agg(["mean", "count"]).sort_values("mean")
print("Signal B -- CTR vs position (position_tier):")
print(bucket_pos)


Signal B -- CTR vs position (position_tier):
                   mean  count
position_tier                 
deep           0.150212   1319
page_3_5       0.222484   7242
striking       0.323239   7304
page_1         0.652467  11814
top_3          1.483611   2321


**Verdict: CONFIRMED.** CTR rises cleanly with position — `deep` (0.15%) → `page_3_5` (0.22%)
→ `striking` (0.32%) → `page_1` (0.65%) → `top_3` (1.48%). This one holds and becomes the
Week-4 baseline rule's backbone.


## 2. Baseline — the rule this capstone's model must beat

From Week 4: flag visible pages (≥500 impressions) whose CTR is under half their position
tier's typical rate. No fitted weights, fully explainable, one reason code.


In [4]:
expected_ctr_for_tier = df.groupby("position_tier")["ctr"].transform("mean")
underperforms_ctr = (df["ctr"] < expected_ctr_for_tier * 0.5).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = underperforms_ctr * visible * df["impressions_90d"]

print("rows scored > 0 by the baseline rule:", (df["baseline_score"] > 0).sum(), "of", len(df))


rows scored > 0 by the baseline rule: 9781 of 30000


## 3. A real leak, found and proven — not assumed

Before building any features: does the dataset's own "last 30 days vs. prior 30 days" columns
directly construct the label? Tested, not guessed.


In [5]:
mask = df["impressions_prev_30d"] > 0
implied_pct_change = (df.loc[mask, "impressions_last_30d"] - df.loc[mask, "impressions_prev_30d"]) / df.loc[mask, "impressions_prev_30d"] * 100
leak_corr = implied_pct_change.corr(df.loc[mask, "trend_pct"])
print(f"correlation between implied impressions pct change and trend_pct: {leak_corr:.6f}")
print("-> trend_pct IS this ratio. impressions_last_30d / impressions_prev_30d excluded from")
print("   features, and the same-family clicks/sessions last_30d & prev_30d columns excluded too.")


correlation between implied impressions pct change and trend_pct: 1.000000
-> trend_pct IS this ratio. impressions_last_30d / impressions_prev_30d excluded from
   features, and the same-family clicks/sessions last_30d & prev_30d columns excluded too.


## 4. Features, split design, and honest framing

Features are 90-day aggregates plus static content attributes — all knowable without the
label's own 30-day comparison window. These aggregates likely still *partially* overlap the
label's most recent 30 days, so this model is framed as a **current-state diagnostic**
("does this page's present profile look like other declining pages?"), not a clean forward
forecast — decision-support language, not causal or predictive-certainty language.

**Split: grouped by `client_id`**, 70/30, so no client's pages leak between train and test.


In [6]:
from sklearn.model_selection import GroupShuffleSplit

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count", "content_age_days",
    "age_tier_order", "days_since_last_update", "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
categorical_features = ["competition_level", "content_type", "main_intent"]
for c in ["word_count", "char_count"]:
    df[f"has_{c}"] = df[c].notna().astype(int)
numeric_features += ["has_word_count", "has_char_count"]

X = df[numeric_features + categorical_features].copy()
y = df["is_declining"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
ytr, yte = y.iloc[train_idx], y.iloc[test_idx]

print("train rows:", len(Xtr), "| test rows:", len(Xte))
print("client overlap between train and test:", len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))


train rows: 19166 | test rows: 10834
client overlap between train and test: 0


## 5. Model comparison — same split, same metric, against the same baseline

Logistic Regression and Decision Tree for readability, Random Forest and Gradient Boosting for
strength — the toolkit menu from Week 5's session. All four scored on the exact same held-out
test fold as the baseline, at precision@K and recall@K (K = top 10% of the test fold) and
ROC-AUC.


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

pre_scaled = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])
pre_unscaled = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])

models = {
    "logistic_regression": Pipeline([("pre", pre_scaled), ("clf", LogisticRegression(max_iter=2000, random_state=42))]),
    "decision_tree_depth3": Pipeline([("pre", pre_unscaled), ("clf", DecisionTreeClassifier(max_depth=3, random_state=42))]),
    "random_forest": Pipeline([("pre", pre_unscaled), ("clf", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1))]),
    "gradient_boosting": Pipeline([("pre", pre_unscaled), ("clf", GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42))]),
}

def precision_recall_at_k(scores, y_true, k):
    order = np.argsort(-scores)
    top_k = order[:k]
    y_true_arr = np.asarray(y_true)
    hits = y_true_arr[top_k].sum()
    return hits / k, hits / y_true_arr.sum()

K = int(0.10 * len(Xte))
baseline_test = df.loc[Xte.index, "baseline_score"].values

rows = []
p, r = precision_recall_at_k(baseline_test, yte, K)
rows.append({"method": "baseline_rule (Week 4)", "precision_at_K": round(p,3), "recall_at_K": round(r,3), "roc_auc": round(roc_auc_score(yte, baseline_test),3)})

fitted_models = {}
for name, pipe in models.items():
    pipe.fit(Xtr, ytr)
    fitted_models[name] = pipe
    proba = pipe.predict_proba(Xte)[:, 1]
    p, r = precision_recall_at_k(proba, yte, K)
    rows.append({"method": name, "precision_at_K": round(p,3), "recall_at_K": round(r,3), "roc_auc": round(roc_auc_score(yte, proba),3)})

comparison = pd.DataFrame(rows)
print(f"K = {K} (top 10% of test fold) | test-fold base rate = {yte.mean():.3f}")
comparison


K = 1083 (top 10% of test fold) | test-fold base rate = 0.559


,method,precision_at_K,recall_at_K,roc_auc
0,baseline_rule (Week 4),0.596,0.107,0.554
1,logistic_regression,0.681,0.122,0.601
2,decision_tree_depth3,0.604,0.108,0.573
3,random_forest,0.657,0.117,0.617
4,gradient_boosting,0.689,0.123,0.632


**Gradient Boosting wins on both metrics** (precision@K 0.689, ROC-AUC 0.632) — earns its
extra complexity over the Decision Tree, which barely beats the baseline (0.604 vs. 0.596),
exactly the "simplicity is a feature, add complexity only when it earns it" rule from the
skill. Gradient Boosting is the model carried forward.


## 6. Reading the errors, not just the score

Permutation importance on the winning model, then concrete wrong cases.


In [8]:
from sklearn.inspection import permutation_importance

gb = fitted_models["gradient_boosting"]
perm = permutation_importance(gb, Xte, yte, n_repeats=8, random_state=42, n_jobs=-1, scoring="roc_auc")
feat_names = numeric_features + categorical_features
importance_df = pd.DataFrame({
    "feature": feat_names,
    "importance": perm.importances_mean,
}).sort_values("importance", ascending=False)

importance_df.head(10)


,feature,importance
16,days_with_impressions,0.084151
18,ctr,0.019554
5,content_age_days,0.013540
19,avg_position,0.011855
21,scroll_rate,0.008243
3,word_count,0.008040
12,users_90d,0.006374
17,days_with_sessions,0.004127
9,clicks_90d,0.003916
8,impressions_90d,0.003824


**Top feature: `days_with_impressions`** (sustained visibility) — sensible, not
suspiciously perfect. `ctr` and `content_age_days` follow, consistent with the two signal checks
in section 1. No single feature dominates the way a leaked column would.


In [9]:
gb_proba_test = gb.predict_proba(Xte)[:, 1]
test_df = df.loc[Xte.index].copy()
test_df["proba"] = gb_proba_test
test_df["pred"] = (gb_proba_test >= 0.5).astype(int)
test_df["actual"] = yte.values

review_cols = ["content_id", "content_type", "avg_position", "ctr", "impressions_90d", "content_age_days", "proba"]
false_neg = test_df[(test_df["actual"] == 1) & (test_df["pred"] == 0)].sort_values("proba").head(2)
false_pos = test_df[(test_df["actual"] == 0) & (test_df["pred"] == 1)].sort_values("proba", ascending=False).head(2)

print("FALSE NEGATIVES (actually declining, model said no):")
print(false_neg[review_cols].to_string(index=False))
print()
print("FALSE POSITIVES (not declining, model said yes):")
print(false_pos[review_cols].to_string(index=False))


FALSE NEGATIVES (actually declining, model said no):
          content_id    content_type  avg_position  ctr  impressions_90d  content_age_days    proba
content_7bc32bc1df59 keyword article           0.0  0.0                1               238 0.035997
content_47d62e68cd7a keyword article          20.0  0.0                2               309 0.095804

FALSE POSITIVES (not declining, model said yes):
          content_id       content_type  avg_position  ctr  impressions_90d  content_age_days    proba
content_9c128be31943 comparison article          23.8 0.15             1296               203 0.906362
content_f79387f83703 comparison article          12.4 0.15             1945               221 0.905907


**Why these are hard:** both error types cluster around near-zero-click content
(`impressions_90d` at or near single digits, `ctr = 0`) — there's genuinely little signal to
separate "just launched, no clicks yet" from "declined to near-zero." This is the model's real
blind spot: low-signal pages, not any one feature or content tier.


## 7. The ranked action engine — reason codes, not just a score

The deployed queue scores every page (not just the test fold — this is the operational pass,
not a performance claim; the performance claim is section 5's held-out table only). Reason codes
are simple, explainable, and priority-ordered from the real findings above, not opaque.


In [10]:
df["model_score"] = gb.predict_proba(X)[:, 1]

REVIEW_THRESHOLD = df["model_score"].quantile(0.90)
df["action"] = np.where(df["model_score"] >= REVIEW_THRESHOLD, "review_declining_risk", "monitor")

def assign_reason(row, expected_ctr):
    if row["impressions_90d"] < 10:
        return "near_zero_signal"
    if row["ctr"] < expected_ctr.loc[row.name] * 0.5:
        return "ctr_below_position_expectation"
    if row["content_age_days"] < 90:
        return "young_and_volatile"
    if row["days_with_impressions"] < df["days_with_impressions"].quantile(0.25):
        return "low_active_days"
    return "model_flagged_review"

flagged = df["action"] == "review_declining_risk"
df.loc[flagged, "reason_code"] = df.loc[flagged].apply(lambda r: assign_reason(r, expected_ctr_for_tier), axis=1)
df.loc[~flagged, "reason_code"] = "not_flagged"

print("rows flagged for review:", flagged.sum(), "of", len(df))
print(df.loc[flagged, "reason_code"].value_counts())


rows flagged for review: 3000 of 30000
reason_code
ctr_below_position_expectation    2384
model_flagged_review               612
near_zero_signal                     3
low_active_days                      1
Name: count, dtype: int64


In [11]:
import os
os.makedirs("../outputs", exist_ok=True)

out_cols = ["content_id", "content_type", "position_tier", "avg_position", "ctr", "impressions_90d",
            "content_age_days", "model_score", "action", "reason_code"]
ranked = df.sort_values("model_score", ascending=False)[out_cols]
ranked.to_csv("../outputs/capstone_ranked_action_engine.csv", index=False)
print("written: work/outputs/capstone_ranked_action_engine.csv")
ranked.head(10)


written: work/outputs/capstone_ranked_action_engine.csv


,content_id,content_type,position_tier,avg_position,ctr,impressions_90d,content_age_days,model_score,action,reason_code
23022,content_0e9a3474ec41,keyword article,top_3,2.1,0.00,160,95,0.951534,review_declining_risk,ctr_below_position_expectation
25248,content_565c92d02525,keyword article,page_3_5,22.4,0.33,2741,223,0.944770,review_declining_risk,model_flagged_review
16578,content_e115a063d3c8,keyword article,page_1,3.9,0.00,762,223,0.938208,review_declining_risk,ctr_below_position_expectation
14992,content_b005f46f2e4c,keyword article,top_3,2.7,0.15,673,165,0.938175,review_declining_risk,ctr_below_position_expectation
3919,content_bcc95460460e,keyword article,top_3,2.5,0.35,6315,230,0.936731,review_declining_risk,ctr_below_position_expectation
29354,content_1f7638fecdc1,keyword article,top_3,2.3,0.00,565,223,0.936505,review_declining_risk,ctr_below_position_expectation
15265,content_6dd02d2e1d19,keyword article,top_3,2.3,0.00,278,165,0.935200,review_declining_risk,ctr_below_position_expectation
24990,content_f6381c4c86c0,keyword article,page_1,7.7,0.00,243,90,0.934522,review_declining_risk,ctr_below_position_expectation
14115,content_5cda32af644c,keyword article,top_3,3.0,0.00,720,148,0.934284,review_declining_risk,ctr_below_position_expectation
15463,content_730c6b20e33b,keyword article,striking,16.2,0.09,6621,96,0.934219,review_declining_risk,ctr_below_position_expectation


## 8. Limitations — stated plainly

- **Scope:** this capstone runs on the 30k-row anonymized starter sample, not the full ~79M-row
  Hugging Face warehouse. Directional findings (e.g. "newer content declines more") may not hold
  at full scale or across all 32 clients equally.
- **Window overlap:** the 90-day aggregate features likely share some calendar time with the
  label's own most recent 30 days. This model is a **current-state diagnostic**, not a clean
  past-predicts-future forecast — its claims are decision-support, not causal.
- **Client imbalance:** history depth varies by client (see Week 3's `dim_clients` check on the
  warehouse); this starter sample doesn't expose that column directly, so per-client fairness of
  the model is unverified here.
- **No causal claim:** nothing here proves *why* a page is declining, or that any recommended fix
  would reverse it — only that its current profile resembles other declining pages, observed and
  directional, not causal.


## 9. Ranked recommendations — the action playbook

1. **`ctr_below_position_expectation`** (2,384 flagged rows) — highest-volume, most actionable
   reason: title/meta-description review for pages ranking well but under-clicking their tier.
2. **`model_flagged_review`** (612 flagged rows) — no single obvious cause; route to manual
   editorial review rather than an automated fix.
3. **`young_and_volatile`** / **`near_zero_signal`** / **`low_active_days`** (rare in this run) —
   watch-list only; too little signal yet for a confident action.

Priority order for an editor's queue: start with `ctr_below_position_expectation` (clear,
explainable, highest volume), then `model_flagged_review` (needs a human look), then treat the
remaining reason codes as a monitoring list, not an action list.
